# [Super AI Engineer Season 6] Hackathon Week 6
## 5 Domains Hackathon: Thai Math VQA Challenge

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Thai Math VQA Challenge
- Notebook: Fast cached VLM inference pipeline
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Configuration  
3. Data Loading & Initial Inspection  
4. Answer Normalization & Metric  
5. Prompting & Answer Extraction  
6. Model Loading  
7. Inference Functions  
8. Quick Validation  
9. Full Test Inference With Cache  
10. Prediction & Submission Generation  
11. Diagnostics & Next Experiments

# 1. Setup & Imports
### 1.1 Prepare dependencies for VQA inference

Install only missing packages, then import the libraries needed for image loading, model inference, normalization, caching, and submission generation.

In [ ]:
import importlib.util
import os
import random
import re
import subprocess
import sys
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict


def has_module(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

INSTALL_DEPS = os.environ.get("MATH_VQA_INSTALL_DEPS", "1") == "1"

if INSTALL_DEPS:
    base_packages = ["accelerate", "qwen-vl-utils", "sentencepiece", "bitsandbytes"]
    missing = [pkg for pkg in base_packages if not has_module(pkg.replace("-", "_"))]
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *missing])

    try:
        from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
    except Exception:
        print("Installing latest Transformers for Qwen2.5-VL support...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "git+https://github.com/huggingface/transformers",
        ])

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Configuration
### 2.1 Set dataset paths, model choices, and runtime controls

Prefer the Kaggle dataset path, fall back to the local repository dataset, and keep model candidates plus cache paths configurable from environment variables.

In [ ]:
DATASET_SLUG = "super-ai-engineer-ss-6-individual-test-thai-math-vqa-challen"
KAGGLE_MODE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working") if KAGGLE_MODE else Path("Level 2/Hackathon 9_5 Domains Hackathon/Math VQA Challenge")
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR_CANDIDATES = []
if os.environ.get("MATH_VQA_DATA_DIR"):
    DATA_DIR_CANDIDATES.append(Path(os.environ["MATH_VQA_DATA_DIR"]))
DATA_DIR_CANDIDATES.extend([
    Path("/kaggle/input/competitions") / DATASET_SLUG,
    Path("/kaggle/input") / DATASET_SLUG,
    Path("Level 2/Hackathon 9_5 Domains Hackathon/Math VQA Challenge") / DATASET_SLUG,
    Path(DATASET_SLUG),
    Path("."),
])


def find_dataset_dir(candidates):
    for candidate in candidates:
        if (candidate / "train.csv").exists() and (candidate / "test.csv").exists() and (candidate / "sample_submission.csv").exists():
            return candidate
    print("Checked dataset candidates:")
    for candidate in candidates:
        print(" -", candidate)
    raise FileNotFoundError("Cannot find Math VQA dataset. Set MATH_VQA_DATA_DIR or attach the Kaggle dataset.")


DATA_DIR = find_dataset_dir(DATA_DIR_CANDIDATES)

# Deadline mode is the default because VLM inference is the bottleneck.
# Override MATH_VQA_FAST_MODE=0 only when there is enough GPU time.
FAST_MODE = os.environ.get("MATH_VQA_FAST_MODE", "1") == "1"
default_primary_model = "Qwen/Qwen2.5-VL-3B-Instruct" if FAST_MODE else "Qwen/Qwen2.5-VL-7B-Instruct"
default_fallback_models = "Qwen/Qwen2.5-VL-3B-Instruct" if FAST_MODE else "Qwen/Qwen2.5-VL-3B-Instruct,Qwen/Qwen2-VL-7B-Instruct"

PRIMARY_MODEL_ID = os.environ.get("MATH_VQA_MODEL_ID", default_primary_model)
FALLBACK_MODEL_IDS = [m.strip() for m in os.environ.get("MATH_VQA_FALLBACK_MODELS", default_fallback_models).split(",") if m.strip()]
LOCAL_MODEL_DIR = os.environ.get("MATH_VQA_MODEL_DIR", "").strip()
MODEL_CANDIDATES = [LOCAL_MODEL_DIR] if LOCAL_MODEL_DIR else []
MODEL_CANDIDATES += [PRIMARY_MODEL_ID] + [m for m in FALLBACK_MODEL_IDS if m != PRIMARY_MODEL_ID]

RUN_MODEL = os.environ.get("MATH_VQA_RUN_MODEL", "1") == "1"
ALLOW_FALLBACK_SUBMISSION = os.environ.get("MATH_VQA_ALLOW_FALLBACK", "1") == "1"
USE_4BIT = os.environ.get("MATH_VQA_USE_4BIT", "1") == "1"
USE_FLASH_ATTN = os.environ.get("MATH_VQA_USE_FLASH_ATTN", "0") == "1"
SELF_CONSISTENCY_N = int(os.environ.get("MATH_VQA_SC_N", "1" if FAST_MODE else "3"))
SELF_CONSISTENCY_TEMP = float(os.environ.get("MATH_VQA_SC_TEMP", "0.35" if FAST_MODE else "0.65"))
MAX_NEW_TOKENS = int(os.environ.get("MATH_VQA_MAX_NEW_TOKENS", "96" if FAST_MODE else "384"))
VALIDATION_N = int(os.environ.get("MATH_VQA_VALIDATION_N", "0" if FAST_MODE else "12"))
SAVE_EVERY = int(os.environ.get("MATH_VQA_SAVE_EVERY", "2" if FAST_MODE else "10"))
MAX_TEST_ROWS = int(os.environ.get("MATH_VQA_MAX_TEST_ROWS", "0"))  # 0 = all rows
ENABLE_RESCUE = os.environ.get("MATH_VQA_ENABLE_RESCUE", "0" if FAST_MODE else "1") == "1"
INFERENCE_TIME_BUDGET_MIN = float(os.environ.get("MATH_VQA_TIME_BUDGET_MIN", "42" if FAST_MODE else "0"))
STOP_MARGIN_SECONDS = float(os.environ.get("MATH_VQA_STOP_MARGIN_SECONDS", "180" if FAST_MODE else "60"))
FAST_MAX_PIXELS = int(os.environ.get("MATH_VQA_FAST_MAX_PIXELS", str(640 * 28 * 28)))
FULL_MAX_PIXELS = int(os.environ.get("MATH_VQA_FULL_MAX_PIXELS", str(1280 * 28 * 28)))
PROCESSOR_MAX_PIXELS = FAST_MAX_PIXELS if FAST_MODE else FULL_MAX_PIXELS
PROCESSOR_MIN_PIXELS = int(os.environ.get("MATH_VQA_MIN_PIXELS", str(128 * 28 * 28 if FAST_MODE else 256 * 28 * 28)))

PREDICTION_CACHE_PATH = WORK_DIR / "math_vqa_v2_predictions.csv"
SUBMISSION_PATH = WORK_DIR / "submission_v2.csv"
BASELINE_SUBMISSION_PATH = WORK_DIR / "submission_v2_baseline.csv"

print("KAGGLE_MODE:", KAGGLE_MODE)
print("DATA_DIR:", DATA_DIR.resolve())
print("WORK_DIR:", WORK_DIR.resolve())
print("FAST_MODE:", FAST_MODE)
print("MODEL_CANDIDATES:", MODEL_CANDIDATES)
print("RUN_MODEL:", RUN_MODEL)
print("USE_4BIT:", USE_4BIT)
print("SELF_CONSISTENCY_N:", SELF_CONSISTENCY_N)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)
print("VALIDATION_N:", VALIDATION_N)
print("TIME_BUDGET_MIN:", INFERENCE_TIME_BUDGET_MIN)


# 3. Data Loading & Initial Inspection
### 3.1 Load metadata and resolve image paths

Read `train.csv`, `test.csv`, and `sample_submission.csv`, then map every row to an image path and build a safe fallback answer from the training distribution.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})
sample_sub["answer"] = sample_sub["answer"].astype(str)
train["answer"] = train["answer"].astype(str)


def resolve_image_path(image_path, data_dir=DATA_DIR):
    raw = str(image_path)
    name = Path(raw).name
    candidates = [
        data_dir / raw,
        data_dir / name,
        data_dir / "images" / name,
        data_dir / "images" / "images" / name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    hits = list(data_dir.rglob(name))
    return hits[0] if hits else None

train["resolved_image_path"] = train["image_path"].map(resolve_image_path)
test["resolved_image_path"] = test["image_path"].map(resolve_image_path)

most_common_answer = train["answer"].mode().iloc[0]
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_sub.shape)
print("Missing train images:", int(train["resolved_image_path"].isna().sum()))
print("Missing test images:", int(test["resolved_image_path"].isna().sum()))
print("IDs match sample/test:", set(sample_sub["id"]) == set(test["id"]))
print("Most common answer:", most_common_answer)
print("Top answers:")
print(train["answer"].value_counts().head(15).to_string())
display(train.head())

# 4. Answer Normalization & Metric
### 4.1 Normalize answers for scoring and submission

Convert Thai digits, remove unnecessary wrappers, standardize numeric answers, and keep final answers short enough for reliable evaluation.

In [ ]:
THAI_DIGITS = "".join(chr(0x0E50 + i) for i in range(10))
THAI_DIGIT_TABLE = str.maketrans(THAI_DIGITS, "0123456789")

EN_UNIT_WORDS = [
    "square centimeters", "square centimetre", "square centimeter", "sqcm", "sq.cm",
    "cubic centimeters", "cubic centimetre", "centimeters", "centimetres", "centimeter", "cm",
    "degrees", "degree", "baht", "dollars", "years old", "years", "year", "units", "unit",
]


def expand_latex(text: str) -> str:
    s = str(text)
    for _ in range(6):
        ns = re.sub(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"\1/\2", s)
        if ns == s:
            break
        s = ns
    s = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", s)
    s = re.sub(r"\\sqrt\s*([0-9a-zA-Z]+)", r"sqrt(\1)", s)
    replacements = {
        r"\pi": "pi", r"\times": "*", r"\cdot": "*", r"\div": "/",
        r"\left": "", r"\right": "", r"\,": "", r"\;": "", r"\!": "",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    return s


def looks_math_like(s: str) -> bool:
    return bool(re.search(r"[0-9]|sqrt|pi|/|\*|\^|=", s, flags=re.IGNORECASE))


def normalize_answer(value) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip().lower()
    if s in {"", "nan", "none", "null", "<n/a>"}:
        return ""
    s = s.translate(THAI_DIGIT_TABLE)
    s = s.replace("$", "")
    s = expand_latex(s)
    s = s.replace("?", "-").replace("?", "*").replace("?", "/")
    for unit in EN_UNIT_WORDS:
        s = re.sub(re.escape(unit), "", s, flags=re.IGNORECASE)
    if looks_math_like(s):
        s = re.sub(r"[\u0E00-\u0E7F]+", "", s)
    s = re.sub(r"[\s{}\\,]", "", s)
    s = s.strip("`*'\"?.!:;")
    s = re.sub(r"^\((-?\d+)\)$", r"\1", s)
    if re.fullmatch(r"[+-]?\d+(?:\.0+)?", s):
        try:
            return str(int(float(s)))
        except Exception:
            return s
    if re.fullmatch(r"[+-]?\d+\.\d+", s):
        return s.rstrip("0").rstrip(".")
    return s


def normalized_accuracy(y_true, y_pred) -> float:
    y_true_norm = [normalize_answer(x) for x in y_true]
    y_pred_norm = [normalize_answer(x) for x in y_pred]
    return float(np.mean([a == b for a, b in zip(y_true_norm, y_pred_norm)]))

fallback_norm = normalize_answer(most_common_answer)
print("fallback raw:", most_common_answer)
print("fallback norm:", fallback_norm)
for example in ["20 square centimeters", "$6\\sqrt{3}$", "$\\frac{17}{10}$", THAI_DIGITS[2] + THAI_DIGITS[5], "2.0", "48\\pi"]:
    print(f"{example:28s} -> {normalize_answer(example)}")

# 5. Prompting & Answer Extraction
### 5.1 Create the prompt and answer extractor

Ask the model for one final answer in a clear tag format, then use fallback extraction patterns when the model does not follow the format exactly.

In [ ]:
ANSWER_FORMAT_HINTS = ", ".join(train["answer"].astype(str).drop_duplicates().head(25).tolist())

BASE_PROMPT = f"""
You are an expert at Thai mathematical word problems and visual reasoning.
Read the image carefully. It may contain Thai text, tables, diagrams, geometry, algebra, arithmetic, or LaTeX-like notation.

Solve the problem internally, then output exactly one final answer in this format:
<answer>FINAL_ANSWER</answer>

Rules for FINAL_ANSWER:
- Use Arabic digits 0-9.
- Do not include units unless the answer is truly a Thai phrase.
- Fractions should look like 17/10.
- Square roots should look like sqrt(3) or 6sqrt(3).
- Pi should look like pi, e.g. 48pi.
- For multiple choice, output the selected value, not a long explanation.
- Keep only the final answer inside the tag.

Known answer styles from training data: {ANSWER_FORMAT_HINTS}
""".strip()

RESCUE_PROMPT = """
Look at the same Thai math image again. Return only the final answer.
Use exactly this format: <answer>FINAL_ANSWER</answer>.
No explanation.
""".strip()

ANSWER_PATTERNS = [
    r"<answer>\s*(.*?)\s*</answer>",
    r"\\boxed\{([^{}]+)\}",
    r"(?:final\s+answer|the\s+answer|answer)\s*(?:is|=|:)?\s*[\"']?([^\n\"'<>]{1,80})",
    r"(?:\u0E04\u0E33\u0E15\u0E2D\u0E1A|\u0E15\u0E2D\u0E1A)\s*(?:\u0E04\u0E37\u0E2D|=|:)?\s*([^\n\"'<>]{1,80})",
]

MATH_CANDIDATE_PATTERN = re.compile(
    r"[+-]?(?:\d+\.\d+|\d+)(?:/\d+)?(?:\s*(?:pi|sqrt\(?\d+\)?))?|sqrt\(?\d+\)?|\d+\s*pi",
    flags=re.IGNORECASE,
)


def clean_candidate(candidate: str) -> str:
    candidate = str(candidate).strip()
    candidate = re.sub(r"</?answer>", "", candidate, flags=re.IGNORECASE).strip()
    candidate = candidate.split("\n")[0].strip()
    candidate = candidate.strip("`*'\"?.!:; ")
    return candidate[:120]


def extract_answer(text: str) -> str:
    if not text:
        return ""
    s = str(text).strip()
    for pattern in ANSWER_PATTERNS:
        m = re.search(pattern, s, re.IGNORECASE | re.DOTALL)
        if m:
            candidate = clean_candidate(m.group(1))
            if candidate:
                return candidate
    candidates = MATH_CANDIDATE_PATTERN.findall(s)
    if candidates:
        return clean_candidate(candidates[-1])
    lines = [clean_candidate(line) for line in s.splitlines() if clean_candidate(line)]
    return lines[-1] if lines else ""

print("Base prompt length:", len(BASE_PROMPT))
print(BASE_PROMPT[:500])

# 6. Model Loading
### 6.1 Load the vision-language model with fallbacks

Try model candidates in order, use quantization when supported, and keep a baseline submission path available if no model can be loaded.

In [ ]:
model = None
processor = None
model_source_used = None
model_load_error = None


def import_vlm_classes():
    from transformers import AutoProcessor
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration as VLMClass
        return AutoProcessor, VLMClass, "Qwen2.5-VL"
    except Exception:
        try:
            from transformers import Qwen2VLForConditionalGeneration as VLMClass
            return AutoProcessor, VLMClass, "Qwen2-VL"
        except Exception:
            from transformers import AutoModelForImageTextToText as VLMClass
            return AutoProcessor, VLMClass, "AutoModelForImageTextToText"


def bitsandbytes_config():
    if not USE_4BIT:
        return None
    try:
        import torch
        if not torch.cuda.is_available():
            return None
        from transformers import BitsAndBytesConfig
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
    except Exception as exc:
        print("4-bit unavailable:", repr(exc))
        return None


def load_model_from_candidates(candidates):
    import torch
    AutoProcessor, VLMClass, class_name = import_vlm_classes()
    print("VLM class:", class_name)
    quant_config = bitsandbytes_config()

    last_error = None
    for candidate in candidates:
        if not candidate:
            continue
        try:
            print("Trying model:", candidate)
            processor_kwargs = dict(trust_remote_code=True)
            try:
                processor_kwargs.update(min_pixels=PROCESSOR_MIN_PIXELS, max_pixels=PROCESSOR_MAX_PIXELS)
            except Exception:
                pass
            proc = AutoProcessor.from_pretrained(candidate, **processor_kwargs)

            kwargs = dict(
                trust_remote_code=True,
                device_map="auto" if torch.cuda.is_available() else None,
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            )
            if quant_config is not None:
                kwargs["quantization_config"] = quant_config
            if USE_FLASH_ATTN:
                kwargs["attn_implementation"] = "flash_attention_2"
            else:
                kwargs["attn_implementation"] = "eager"

            mdl = VLMClass.from_pretrained(candidate, **kwargs)
            mdl.eval()
            print("Loaded model:", candidate)
            print("Device:", next(mdl.parameters()).device)
            if torch.cuda.is_available():
                print("GPU memory GB:", round(torch.cuda.memory_allocated() / 1e9, 2))
            return proc, mdl, candidate, None
        except Exception as exc:
            last_error = exc
            print("Failed:", candidate, "->", repr(exc))
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass
    return None, None, None, last_error

if RUN_MODEL:
    processor, model, model_source_used, model_load_error = load_model_from_candidates(MODEL_CANDIDATES)
else:
    model_load_error = RuntimeError("RUN_MODEL is disabled by MATH_VQA_RUN_MODEL=0")

if model is None:
    print("Model is not available:", repr(model_load_error))
    if not ALLOW_FALLBACK_SUBMISSION:
        raise RuntimeError("Model could not be loaded and fallback submission is disabled.")
    print("Continuing with baseline fallback so submission format can still be produced.")

# 7. Inference Functions
### 7.1 Run image reasoning and choose the final answer

Combine deterministic decoding with optional self-consistency, normalize every candidate answer, and select the most reliable final response.

In [ ]:
def run_single_inference(image_path: Path, prompt_text: str, temperature: float = 0.0) -> str:
    if model is None or processor is None:
        return ""

    import torch
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": str(image_path)},
            {"type": "text", "text": prompt_text},
        ],
    }]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    try:
        from qwen_vl_utils import process_vision_info
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
    except Exception:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")

    device = next(model.parameters()).device
    inputs = inputs.to(device)
    gen_kwargs = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=temperature > 0,
        num_beams=1,
        use_cache=True,
    )
    if temperature > 0:
        gen_kwargs.update(temperature=temperature, top_p=0.90)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, **gen_kwargs)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]


def choose_best_answer(answers_raw):
    answers_norm = [normalize_answer(x) for x in answers_raw]
    valid = [x for x in answers_norm if x]
    if not valid:
        return fallback_norm, "", 0.0, answers_norm
    counts = Counter(valid)
    best_norm, votes = counts.most_common(1)[0]
    best_raw = next((raw for raw, norm in zip(answers_raw, answers_norm) if norm == best_norm), best_norm)
    confidence = votes / max(len(answers_norm), 1)
    return best_norm, best_raw, confidence, answers_norm


def run_self_consistency(image_path: Path, n: int = SELF_CONSISTENCY_N) -> dict:
    n = max(int(n), 1)
    prompts = [BASE_PROMPT]
    if n > 1:
        prompts += [BASE_PROMPT, RESCUE_PROMPT]
    answers_raw = []
    raw_outputs = []

    for i in range(n):
        prompt = prompts[min(i, len(prompts) - 1)]
        temperature = 0.0 if i == 0 else SELF_CONSISTENCY_TEMP
        try:
            raw_output = run_single_inference(image_path, prompt, temperature=temperature)
            answer_raw = extract_answer(raw_output)
            if ENABLE_RESCUE and not normalize_answer(answer_raw):
                rescue = run_single_inference(image_path, RESCUE_PROMPT, temperature=0.0)
                raw_output = raw_output + "\n\n[RESCUE]\n" + rescue
                answer_raw = extract_answer(rescue)
        except Exception as exc:
            raw_output = "ERROR:" + repr(exc)
            answer_raw = ""
        raw_outputs.append(raw_output)
        answers_raw.append(answer_raw)

    best_norm, best_raw, confidence, answers_norm = choose_best_answer(answers_raw)
    return {
        "outputs": raw_outputs,
        "answers_raw": answers_raw,
        "answers_norm": answers_norm,
        "best_answer_raw": best_raw,
        "best_answer_norm": best_norm,
        "confidence": confidence,
    }

print("Inference functions ready. Model available:", model is not None)
print("Rescue pass enabled:", ENABLE_RESCUE)


# 8. Quick Validation
### 8.1 Run a small validation check

Use a small subset of training images to verify that the model, prompt, and normalizer work before spending time on the full test set.

In [ ]:
from sklearn.model_selection import train_test_split

train_meta, valid_meta = train_test_split(train, test_size=0.2, random_state=SEED)
baseline_score = normalized_accuracy(valid_meta["answer"], [most_common_answer] * len(valid_meta))
print("Most common answer:", most_common_answer)
print("Baseline validation accuracy:", round(baseline_score, 4))

val_df = pd.DataFrame()
if model is not None and VALIDATION_N > 0:
    available_valid = valid_meta.dropna(subset=["resolved_image_path"])
    val_n = min(VALIDATION_N, len(available_valid))
    val_sample = available_valid.sample(val_n, random_state=SEED)
    val_rows = []
    for _, row in tqdm(val_sample.iterrows(), total=val_n, desc="Validation inference"):
        result = run_self_consistency(Path(row["resolved_image_path"]), n=SELF_CONSISTENCY_N)
        true_norm = normalize_answer(row["answer"])
        pred_norm = result["best_answer_norm"] or fallback_norm
        val_rows.append({
            "id": row["id"],
            "true_answer": row["answer"],
            "true_norm": true_norm,
            "pred_norm": pred_norm,
            "confidence": result["confidence"],
            "correct": true_norm == pred_norm,
            "raw_answers": str(result["answers_raw"]),
        })
    val_df = pd.DataFrame(val_rows)
    print("Validation sample accuracy:", round(float(val_df["correct"].mean()), 4))
    display(val_df[["id", "true_norm", "pred_norm", "confidence", "correct"]])
else:
    print("Skipping VLM validation because model is unavailable or VALIDATION_N=0.")

# 9. Full Test Inference With Cache
### 9.1 Predict test rows with resume support

Resume from the prediction cache when it exists, run only unfinished rows, and save often so an interrupted session still leaves usable predictions.

In [ ]:
if PREDICTION_CACHE_PATH.exists():
    pred_df = pd.read_csv(PREDICTION_CACHE_PATH, dtype={"id": str})
    done_ids = set(pred_df["id"].astype(str)) if "id" in pred_df.columns else set()
    print("Loaded cache rows:", len(pred_df))
else:
    pred_df = pd.DataFrame()
    done_ids = set()

rows_to_run = test.copy()
if MAX_TEST_ROWS > 0:
    rows_to_run = rows_to_run.head(MAX_TEST_ROWS).copy()

rows_to_run["id"] = rows_to_run["id"].astype(str)
rows_to_run = rows_to_run[~rows_to_run["id"].isin(done_ids)].reset_index(drop=True)
print("Rows still needing inference:", len(rows_to_run))

start_time = time.time()
deadline = None
if INFERENCE_TIME_BUDGET_MIN > 0:
    deadline = start_time + INFERENCE_TIME_BUDGET_MIN * 60
    print("Inference will stop early after about", INFERENCE_TIME_BUDGET_MIN, "minutes to leave time for submission build.")

new_rows = []

def flush_new_rows():
    global pred_df, new_rows
    if not new_rows:
        return
    batch = pd.DataFrame(new_rows)
    pred_df = pd.concat([pred_df, batch], ignore_index=True)
    pred_df = pred_df.drop_duplicates("id", keep="last")
    pred_df.to_csv(PREDICTION_CACHE_PATH, index=False)
    print("Cached rows:", len(pred_df), "->", PREDICTION_CACHE_PATH)
    new_rows = []

for _, row in tqdm(rows_to_run.iterrows(), total=len(rows_to_run), desc="Test inference"):
    if deadline is not None and time.time() >= deadline - STOP_MARGIN_SECONDS:
        print("Stopping early to protect submission time. Current cache rows:", len(pred_df) + len(new_rows))
        break

    row_id = str(row["id"])
    image_path = row["resolved_image_path"]
    if model is None or image_path is None or not Path(image_path).exists():
        result = {
            "best_answer_raw": fallback_norm,
            "best_answer_norm": fallback_norm,
            "confidence": 0.0,
            "answers_norm": [fallback_norm],
        }
    else:
        result = run_self_consistency(Path(image_path), n=SELF_CONSISTENCY_N)
        if not result["best_answer_norm"]:
            result["best_answer_norm"] = fallback_norm
            result["best_answer_raw"] = fallback_norm
            result["confidence"] = 0.0

    new_rows.append({
        "id": row_id,
        "image_path": row["image_path"],
        "best_answer_raw": result["best_answer_raw"],
        "best_answer_norm": result["best_answer_norm"],
        "confidence": result["confidence"],
        "all_answers": str(result["answers_norm"]),
    })

    if len(new_rows) >= SAVE_EVERY:
        flush_new_rows()

flush_new_rows()

print("Prediction cache rows:", len(pred_df))
if len(pred_df):
    print("Average confidence:", round(float(pd.to_numeric(pred_df["confidence"], errors="coerce").fillna(0).mean()), 3))
    display(pred_df.head())


# 10. Prediction & Submission Generation
### 10.1 Merge predictions into the submission template

Fill the sample submission with cached predictions and use the fallback answer for any row that is still missing.

In [ ]:
baseline_sub = sample_sub[["id"]].copy()
baseline_sub["answer"] = fallback_norm
baseline_sub.to_csv(BASELINE_SUBMISSION_PATH, index=False)

sub = sample_sub[["id"]].copy()
sub["id"] = sub["id"].astype(str)

if len(pred_df):
    pred_merged = pred_df[["id", "best_answer_norm"]].copy()
    pred_merged["id"] = pred_merged["id"].astype(str)
    pred_merged = pred_merged.drop_duplicates("id", keep="last")
    sub = sub.merge(pred_merged, on="id", how="left")
    sub["answer"] = sub["best_answer_norm"].fillna(fallback_norm).astype(str)
else:
    sub["answer"] = fallback_norm

sub.loc[sub["answer"].astype(str).str.len() == 0, "answer"] = fallback_norm
sub = sub[["id", "answer"]]

assert list(sub.columns) == ["id", "answer"]
assert len(sub) == len(sample_sub), f"row mismatch: {len(sub)} vs {len(sample_sub)}"
assert set(sub["id"]) == set(sample_sub["id"]), "id mismatch against sample_submission"
assert sub["id"].duplicated().sum() == 0, "duplicate ids"
assert sub["answer"].isna().sum() == 0, "missing answers"
assert (sub["answer"].astype(str).str.len() > 0).all(), "empty answers"

sub.to_csv(SUBMISSION_PATH, index=False)
print("Baseline submission:", BASELINE_SUBMISSION_PATH)
print("Final submission:", SUBMISSION_PATH)
print("Shape:", sub.shape)
print("Unique answers:", sub["answer"].nunique())
print(sub["answer"].value_counts().head(15).to_string())
display(sub.head(20))

# 11. Diagnostics & Next Experiments
### 11.1 Inspect weak predictions

Review low-confidence or fallback-heavy outputs to decide which prompts, models, or manual checks should be tried next.

In [ ]:
if len(pred_df):
    low_conf = pred_df[pred_df["confidence"] < 0.5].sort_values("confidence")
    print("Low-confidence rows:", len(low_conf))
    display(low_conf[["id", "best_answer_norm", "confidence", "all_answers"]].head(20))
else:
    print("No model predictions were generated; final submission is the baseline fallback.")

print("\nFinal summary")
print("Dataset:", DATA_DIR)
print("Model used:", model_source_used)
print("Model load error:", repr(model_load_error) if model is None else "None")
print("Prediction rows:", len(pred_df))
print("Submission path:", SUBMISSION_PATH)
print("Baseline path:", BASELINE_SUBMISSION_PATH)
print("\nNext experiments:")
print("1. Set MATH_VQA_SC_N=5 for slower but stronger self-consistency.")
print("2. Set MATH_VQA_MODEL_ID=Qwen/Qwen2.5-VL-3B-Instruct if 7B is too slow or OOM.")
print("3. Review low-confidence images manually and rerun those rows with a stronger model or larger max_new_tokens.")
print("4. If a local/offline model is attached as a Kaggle dataset, set MATH_VQA_MODEL_DIR to that path.")